为什么一堆线性层、点积、归一化、门控，最后能组成一个会“读上下文、记顺序、做推理”的模型？
## 3.1 注意力机制
如果 Embedding 做的是“把 token 变成向量”，那 Attention 做的就是让每个位置，根据自己的需求，去上下文里动态检索信息。
也就是所谓的 **Scaled Dot-Product Attention**： 

$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

- **Query（Q）**：我现在想找什么？
- **Key（K）**：我这里有什么标签，别人能不能通过我找到我？
- **Value（V）**：如果别人真的关注我，我具体把什么内容交给它？

### 3.1.1 最后一次
这是我最后一次去以下面的例子学习普通点积自注意力。假设句子是：
```
["我", "吃", "苹果"]
```
我们观察最后一个 token —— “苹果”。这时候它面对的歧义是：

- “苹果”可以是水果
- 也可以是 Apple 公司 / 手机

那模型怎么判断？它不会查字典，而是靠上下文做语义对齐（这是在训练中发生的！）。

| 组件 | 问题                | 在“苹果”例子里的直觉      |
| -- | ----------------- | ---------------- |
| Q  | 我现在需要什么信息？        | “我到底是食物还是电子产品？”  |
| K  | 我能提供什么线索？         | “吃”携带“进食动作”的标签   |
| V  | 如果你关注我，我真正给你什么内容？ | “吃”会把“食物相关语义”传过来 |

于是计算发生了：
1. “苹果”的 Query 去和全句每个词的 Key 做相似度匹配
2. 它发现自己和“吃”的 Key 很匹配
3. softmax 后，“吃”拿到一部分注意力权重
4. 最后输出向量变成 $ 0.6 \cdot V_{\text{苹果}} + 0.3 \cdot V_{\text{吃}} + \cdots $

### 3.1.2 形状流水线
```
输入隐藏状态 X
[B, S, d_model]

线性投影（三个全局共享的线性层权重）
Q = XW_Q  -> [B, S, d_k]
K = XW_K  -> [B, S, d_k]
V = XW_V  -> [B, S, d_v]

分数矩阵
Q @ K^T   -> [B, S, S]
[
  [我→我,  我→吃,  我→苹果],
  [吃→我,  吃→吃,  吃→苹果],
  [苹果→我, 苹果→吃, 苹果→苹果],
]


softmax
weights   -> [B, S, S]

加权求和
weights @ V -> [B, S, d_v]
```
比如说`我 → 吃 → 苹 → 果`，用前 3 个 token，预测第 4 个 token = 果。
所有 Token Embedding 都是随机初始化的，Wq、Wk、Wv 也是随机初始化，算出来的 Q、K 全是随机向量，`Q @ K^T` 注意力分数完全乱给。
可能 “苹” 的注意力全跑到 “我” 身上，根本不看 “吃”，预测下一个 token 的概率一塌糊涂。
然而模型发现只要让 “对预测有用的 token” 拥有更高的 QK 点积，交叉熵损失就会变小。
于是反向传播时梯度就会自动调整参数 Wq、Wk、Embedding，让下一次预测 “果” 的概率变高，损失变小。
在无数次迭代里，会试出一个最优策略：
想猜对 “果”，必须让 “苹” 重点关注 “吃”想做到这一点，最简单的方法就是：让 “苹” 的 Q 和 “吃” 的 K 点积更大。
于是梯度就会：
- 调整 苹 的 Embedding
- 调整 吃 的 Embedding
- 调整 Wq，让 苹 算出的 Q 更偏向 “食物语义”
- 调整 Wk，让 吃 算出的 K 更偏向 “动作语义”
最终：Q_苹・K_吃 变得很大

从此以后：
- 代表食物 / 物体的 token，Q 会长得很像
- 代表动作 / 吃的 token，K 会长得很像
- 它们的点积天然就大
- 注意力权重自动集中在有用的上下文
### 3.1.3 维度缩放：为什么要除以 $\boldsymbol{\sqrt{d_k}}$？
#### 第一层：点积的方差会随维度 $d_k$ 线性增长
我们先做最基础的数学假设：Query向量 $q$ 和 Key向量 $k$ 中的每一个元素，都是**相互独立、均值为0、方差为1**的随机变量（这也是Transformer权重初始化的默认目标）。

两个独立随机变量的点积计算为：
$$
q \cdot k = \sum_{i=1}^{d_k} q_i \cdot k_i
$$

我们可以严格推导它的均值和方差：
1. 单元素乘积的均值：$\mathbb{E}[q_i \cdot k_i] = \mathbb{E}[q_i] \cdot \mathbb{E}[k_i] = 0 \times 0 = 0$
2. 单元素乘积的方差：$\text{Var}(q_i \cdot k_i) = \mathbb{E}[(q_i k_i)^2] - (\mathbb{E}[q_i k_i])^2 = \mathbb{E}[q_i^2] \cdot \mathbb{E}[k_i^2] = 1 \times 1 = 1$
3. 求和后的总方差：$\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = d_k$

**点积结果的标准差为 $\sqrt{d_k}$，会随着 $d_k$ 的增大而线性增长**。
比如当 $d_k=128$ 时，点积的标准差就达到了 $\sqrt{128} \approx 11.3$，很容易出现绝对值极大的logits值。

#### 第二层：logits 绝对值过大，会让 softmax 进入梯度饱和区
我们先明确softmax的核心公式，对于注意力分数向量 $z = [z_1, z_2, ..., z_S]$，softmax输出为：
$$
\sigma_i = \frac{e^{z_i}}{\sum_{j=1}^S e^{z_j}}
$$

而softmax的雅可比矩阵（梯度传递的核心）为：
$$
\frac{\partial \sigma_i}{\partial z_j} = \sigma_i \left( \delta_{ij} - \sigma_j \right)
$$
其中 $\delta_{ij}$ 为克罗内克函数：当 $i=j$ 时取值为1，否则为0。

当某个logit $z_i$ 绝对值极大时，会出现两种极端情况：
1.  $z_i \gg$ 其他所有 $z_j$：$\sigma_i \approx 1$，其余 $\sigma_j \approx 0$，此时梯度 $\frac{\partial \sigma_i}{\partial z_j} \approx 0$
2.  $z_i \ll$ 其他所有 $z_j$：$\sigma_i \approx 0$，此时梯度 $\frac{\partial \sigma_i}{\partial z_j} \approx 0$

两种情况都会让softmax进入**饱和区**，核心结论是：
> **softmax 一旦饱和，反向传播的梯度会无限趋近于0**。

#### 第三层：梯度消失会直接导致 Q/K 投影参数更新停滞
注意力分数的核心来源是 $QK^\top$，而 $Q$ 和 $K$ 分别来自输入隐藏态与可学习参数 $W_Q$、$W_K$ 的线性投影：
$$
Q = X W_Q, \quad K = X W_K
$$

反向传播的链路是：
`交叉熵损失 ← softmax输出 ← 注意力logits ← QK^T ← W_Q/W_K ← 输入Embedding`

一旦softmax饱和，梯度在这一步就会被“拦腰截断”，回传到 $W_Q$、$W_K$ 的梯度会变得极小，参数更新速度会指数级变慢，模型根本学不到“该关注哪个token、该忽略哪个token”的核心能力。

整个失效链路可以总结为：
> **维度升高 → 点积方差爆炸 → logits绝对值过大 → softmax梯度饱和 → 反向传播梯度消失 → Q/K投影参数更新停滞**

#### 第四层：$\boldsymbol{\sqrt{d_k}}$ 是最朴素且有效的方差校准器
原始Transformer论文中，在softmax之前除以 $\sqrt{d_k}$，本质就是**把点积结果的方差拉回1，让logits始终保持在梯度健康的区间内**。

经过缩放后的点积为：
$$
\frac{q \cdot k}{\sqrt{d_k}}
$$
此时它的方差为：
$$
\text{Var}\left( \frac{q \cdot k}{\sqrt{d_k}} \right) = \frac{1}{d_k} \cdot \text{Var}(q \cdot k) = \frac{1}{d_k} \cdot d_k = 1
$$

这一步缩放有两个核心特性：
1.  **不改变相对相似度**：除以同一个常数，不会改变token之间的相似度排序，只缩放绝对值大小
2.  **稳定训练梯度**：把logits拉回softmax的线性区，避免梯度过早消失，让Q/K参数能持续学习

---

#### 拓展：既然 $\boldsymbol{\sqrt{d_k}}$ 已经能控尺度，为什么还要引入 QK-Norm？
##### 1. 固定缩放的局限性
工业界在超大规模、深层、长上下文模型的训练中发现，仅靠固定的 $\sqrt{d_k}$ 缩放，无法完全解决注意力logits漂移的问题，核心痛点有4个：
- **分布偏移问题**：训练过程中，Q/K向量的分布会随着参数更新发生偏移，不再严格满足“均值0、方差1”的初始假设，固定缩放无法适配动态变化的分布
- **长上下文场景失效**：当序列长度 $S$ 扩展到8K、32K甚至128K时，即使单头方差可控，大量token的点积叠加后依然会出现极值，导致softmax饱和
- **分组注意力（GQA/MQA）的适配问题**：现代大模型普遍使用GQA/MQA降低显存占用，Key/Value的头数远少于Query头数，固定缩放无法适配这种不对称的投影结构
- **深层模型的梯度累积问题**：几十上百层的Transformer堆叠后，每层的微小分布偏移会逐层放大，最终导致注意力logits彻底失控

##### 2. QK-Norm 的核心定义与工业界标准实现
QK-Norm（Query-Key Normalization）的核心思想是：**先对Q和K做逐头的归一化，再用可学习的温度系数替代固定的 $\sqrt{d_k}$ 缩放**，从根源上保证注意力logits的分布始终稳定。

主流工业界实现的标准公式如下：
1.  **逐头层归一化**：对每个注意力头的Q和K分别做LayerNorm，强制拉回均值0、方差1的标准分布
    $$
    \hat{Q} = \text{LayerNorm}(Q), \quad \hat{K} = \text{LayerNorm}(K)
    $$
    注：归一化仅在头维度（$d_k$ 维度）执行，不跨batch、序列和注意力头，保证每个头的分布独立可控。

2.  **可学习温度系数缩放**：用一个可学习的标量参数 $\tau$（初始值通常设为 $\sqrt{d_k}$）替代固定缩放，让模型自主调整注意力的“温度”
    $$
    \text{Attention}(Q,K,V) = \text{softmax}\left( \frac{\hat{Q} \hat{K}^\top}{\tau} \right) V
    $$

部分模型（如Qwen3、Gemma 3）还会做进阶优化：
- 对归一化后的Q/K做L2归一化，让点积等价于余弦相似度，取值范围严格限制在 $[-1, 1]$ 之间，彻底消除维度和序列长度带来的方差影响
- 给温度系数设置上下限，避免模型学出极端的温度值导致训练不稳定
- 对归一化后的Q/K加入极小的权重衰减，进一步约束分布波动

##### 3. QK-Norm 的核心优势
| 优势 | 说明 |
| :--- | :--- |
| 极致的训练稳定性 | 从根源上强制Q/K的分布稳定，即使在100+层、128K长上下文、超大batch的极端场景下，也能避免softmax饱和，彻底解决loss爆炸、训练崩溃的问题 |
| 更好的长上下文适配性 | 归一化后的点积天然等价于余弦相似度，不会随序列长度增长出现极值，在超长上下文场景下依然能保持精准的注意力聚焦 |
| 缓解灾难性遗忘 | 稳定的注意力分布能减少深层模型的参数震荡，提升模型的持续学习能力，在多轮微调、增量预训练场景下表现更优 |
| 更低的微调门槛 | 即使在小batch、低资源的微调场景下，也能保持训练稳定，不会出现梯度消失/爆炸的问题，大幅降低了大模型落地的工程门槛 |

##### 4. 主流大模型的实践落地
| 模型 | 应用说明 |
| :--- | :--- |
| Qwen3 | 技术报告中明确将QK-Norm作为基础架构的核心标配组件，搭配GQA、SwiGLU、RoPE、RMSNorm、Pre-Norm结构，在7B/14B/72B全尺寸模型上实现了训练稳定性和下游效果的双重提升 |
| Gemma 3 | Google在Gemma 3系列模型中引入了改进版的QK-Norm，针对长上下文场景做了L2归一化优化，在128K上下文长度下依然保持了极佳的注意力质量和长文本理解能力 |
| Llama 3.1/3.2 | Meta在长上下文版本的Llama模型中，引入了可选的QK-Norm分支，作为支撑1M超长上下文稳定训练的核心组件 |
| DeepSeek-V2/V3 | 深度求索的开源大模型中，将QK-Norm作为长上下文训练的标配，大幅降低了超长序列下的训练难度，同时提升了长文本任务的效果 |

##### 5. 补充说明
QK-Norm不是对 $\sqrt{d_k}$ 缩放的否定，而是**对它的升级和泛化**：
- 固定 $\sqrt{d_k}$ 是「先验的、静态的」缩放，只能适配权重初始化的理想分布
- QK-Norm是「自适应的、动态的」缩放，能适配训练全过程的分布变化

两者的核心目标始终完全一致：**控制注意力logits的尺度，避免softmax饱和，保证梯度健康传递，让模型能稳定学到有效的注意力模式**。

### 3.1.6 实现思路

In [ ]:
import torch
import math

def stable_softmax(x: torch.Tensor, dim: int=-1) -> torch.Tensor:
    # 先减去最大值来稳定数值
    x_max = x.max(dim=dim, keepdim=True).values
    x_exp = torch.exp(x - x_max)
    return x_exp / x_exp.sum(dim=dim, keepdim=True)

def scaled_dot_product_attention(
    q: torch.Tensor,        # [..., SeqLen_q也就是n, d_k]
    k: torch.Tensor,        # [..., SeqLen_k也就是m, d_k]
    v: torch.Tensor,        # [..., SeqLen_v也是m, d_v]
    mask: torch.Tensor | None = None,
):
    d_k = q.size(-1)        # d_k 是每个头的维度

    # 1. 打分
    scores = torch.einsum("...nd,...md->...nm", q, k)  # [Batch, SeqLen_q, SeqLen_k]
    # 2. mask
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))

    # 3. softmax得到注意力权重
    attn = stable_softmax(scores, dim=-1)  # [Batch, SeqLen_q, SeqLen_k]

    # 4. 加权求和得到输出
    output = torch.einsum("...nm,...md->...nd", attn, v)  # [Batch, SeqLen_q, d_v]
    return output, attn

In [2]:
torch.manual_seed(0)

B, S, d = 1, 3, 2
q = torch.randn(B, S, d)
k = torch.randn(B, S, d)
v = torch.randn(B, S, d)

# 因果 mask：下三角为 True
mask = torch.tril(torch.ones(S, S, dtype=torch.bool))

out_no_mask, attn_no_mask = scaled_dot_product_attention(q, k, v, mask=None)
out_mask, attn_mask = scaled_dot_product_attention(q, k, v, mask=mask)

print("No mask attention:\n", attn_no_mask[0])
print("With causal mask:\n", attn_mask[0])

No mask attention:
 tensor([[0.6601, 0.1685, 0.1714],
        [0.0782, 0.4458, 0.4760],
        [0.0363, 0.6953, 0.2684]])
With causal mask:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.1493, 0.8507, 0.0000],
        [0.0363, 0.6953, 0.2684]])


## 3.2 因果多头自注意力
模型里实际跑的不是“单头、无约束”的注意力，而是 **因果的、多头的、自注意力**。
* **Self**：Q/K/V 都来自同一段输入
* **Causal**：当前位置不能看未来
* **Multi-Head**：不是只用一个视角看上下文，而是多个子空间并行看

### 3.2.1 为什么非得多头？
单头自注意力里，每个 token 只能输出一组加权后的 V 向量，相当于把所有语义信息硬揉在一个向量（**同一个表示子空间**）里，根本没法同时捕捉多种关联。
举个我们之前熟悉的例子：`我用苹果电脑吃苹果`
- 单头注意力里，两个「苹果」的 QK 相似度会完全混在一起
- 模型没法同时抓住「苹果→电脑」的电子产品关联，和「苹果→吃」的食物关联
- 最终只能输出一个「折中混合」的注意力权重，两种语义都抓不准
MHA 的的核心操作非常简单：
1.  把模型隐藏维度 `d_model` 平均拆成 `h` 个独立的头（head），每个头的维度 `d_k = d_model / h`
2.  给每个头分配**独立的、互不共享的 `W_Q/W_K/W_V` 投影矩阵**，每个头都能看到完整的输入隐藏态`[B, S, d_model]`，每个头都有一套完全独立、互不共享、初始值随机的W_Q/W_K/W_V投影矩阵。
3.  每个头**完全独立地算一遍自注意力**，每个头专门学一种语义模式
4.  最后把所有头的输出拼接起来，再用一个线性层投影回 `d_model` 维度
比如8头注意力里：
- 头1：专门学「主谓宾/定状补」语法结构关联
- 头2：专门学「代词→先行词」的指代消解
- 头3：专门学「实体→属性」的常识关联
- 头4：专门学跨句子的长距离依赖
- 头5：专门学否定/转折等逻辑关系
- ... 

> 这听起来人为的刻意设计不太可能，但是实际上多头一般都不会趋同而是分化。
> 每个头的W_Q/W_K/W_V都是独立随机初始化的，初始的线性变换视角完全不同。就像 8 个盲人摸象，每个人初始站的位置完全不一样，摸到的部位不一样，最终形成的对 “大象” 的认知，也绝对不可能完全一样。哪怕训练过程中，它们都在往「降低损失」的方向走，也只会从不同的起点，走到不同的「局部最优解」，每个最优解对应一种不同的语义模式，绝不会趋同。
>
> Transformer 原文也可视化了不同头的注意力分布，发现有的头专门捕捉相邻 token 的语法关联，有的头专门捕捉跨句子的长距离依赖，完全是训练自动形成的，没有任何人工干预。后续大量针对 BERT 的分析论文发现，BERT 的 12 层 ×12 头注意力中，有专门负责指代消解的头、专门负责句法分析的头、专门负责否定词逻辑的头，甚至有专门负责数字关联的头 —— 这些分工全是训练自动形成的。


### 3.2.2 形状流水线
```
输入
x: [B, S, D]

Q/K/V 投影
q, k, v: [B, S, D]

拆头
[B, S, D]
-> [B, S, H, d_head]
-> [B, H, S, d_head]

每个头独立算注意力
scores: [B, H, S, S]
attn:   [B, H, S, S]
out:    [B, H, S, d_head]

合并头
[B, H, S, d_head]
-> [B, S, H, d_head]
-> [B, S, D]

输出投影
[B, S, D]
```

### 3.2.3 长上下文场景的注意力优化方案
#### 3.2.3.1 SRAM内存墙
MHA虽然表达能力强，但在长上下文推理场景下，会遇到无法回避的**内存墙问题**，这也是我们优化注意力架构的核心动机：
1.  **K/V缓存的显存爆炸**：LLM推理时会用KV Cache缓存历史token的K/V向量，避免重复计算。MHA中每个Q头都对应独立的K/V头，序列长度S越大、头数H越多，KV Cache的显存占用就越高（显存占用≈2·B·S·D·层数）。
2.  **SRAM装不下，速度暴跌**：GPU的计算核心（Tensor Core）速度极快，但片上SRAM缓存容量极小（A100单SM的SRAM仅几十MB）。长上下文下，`[B, H, S, S]`的分数矩阵、KV Cache无法塞进SRAM，计算核心需要反复从速度慢一个数量级的HBM显存中搬运数据，90%以上的时间都在“等数据”，推理速度直接暴跌。
3.  **优化核心思路**：在尽量不损失模型效果的前提下，**减少KV头的数量，降低KV Cache的显存占用，让数据能塞进SRAM，减少HBM的反复读写**。  

#### 3.2.3.2 MQA
也即 Multi-Query Attention，多查询注意力。
**核心设计**：**Query保持H个独立头，Key和Value全局共享1个公共头**，所有Q头共用同一组K/V向量。
- 解决的痛点：极致压缩KV Cache的显存占用，把KV Cache的体积直接缩小到原来的1/H。
- 形状流水线核心变化：
  ```python
  # MHA：Q/K/V 均为 [B, H, S, d_head]
  # MQA：Q 保持 [B, H, S, d_head]，K/V 仅为 [B, 1, S, d_head]
  q: [B, H, S, d_head]
  k: [B, 1, S, d_head]  # 头维度=1，全局共享
  v: [B, 1, S, d_head]  # 头维度=1，全局共享

  # 分数计算时，K/V会自动广播到所有Q头，形状仍输出 [B, H, S, S]
  scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)  # [B, H, S, S]
  ```
- **优缺点**：
  - 优势：显存占用最低、推理速度最快，完美适配长上下文场景，KV Cache读写压力极小。
  - 劣势：效果损失明显，相比MHA会出现精度下降，尤其在复杂推理、长文本理解任务上。
- **代表落地模型**：PaLM、Llama 2、StarCoder。
#### 3.2.3.3 GQA
也即 Grouped-Query Attention，分组查询注意力。
**核心设计**：MHA和MQA的折中方案——**把H个Q头分成G个组，每个组共享1个独立的K/V头**。
- 举个例子：H=32个Q头，分成G=8个组，每个组4个Q头，共用1个K/V头，总共有8个K/V头，既压缩了显存，又保留了效果。
- 形状流水线核心变化：
  ```python
  # GQA核心形状
  G = 8  # 分组数，必须满足 H % G == 0
  q: [B, H, S, d_head]          # Q头数保持H=32
  k: [B, G, S, d_head]          # KV头数=分组数G=8
  v: [B, G, S, d_head]          # KV头数=分组数G=8

  # 计算时，每个组内的Q头共享本组的K/V，最终输出形状仍为 [B, H, S, S]
  ```
- **优缺点**：
  - 优势：完美平衡效果与速度，显存占用和推理速度接近MQA，模型效果几乎与MHA持平，是当前工业界的绝对主流方案。
  - 无明显短板，仅需提前设计好分组数，保证H能被G整除。
- **代表落地模型**：Llama 3/3.1/3.2、Qwen全系列、DeepSeek、Gemini、Mistral。
#### 3.2.3.4 MLA
也即 Multi-head Latent Attention，多头隐层注意力。
**核心设计**：Llama 3.1 128K长上下文版本推出的进阶优化方案，在GQA的基础上进一步压缩KV的维度——**把K/V向量压缩到更低维度的隐空间中，推理时再还原，极致降低KV Cache的显存占用**。
- 核心优化：传统MHA/GQA中，K/V的维度和Q一致；MLA中，先把K/V压缩成低维的隐向量存入KV Cache，计算注意力时再通过线性层还原到目标维度，KV Cache的体积可再降低50%以上。
- **优缺点**：
  - 优势：在GQA的基础上进一步降低长上下文的显存占用，128K+超长上下文场景下效果和速度兼顾，是当前最先进的注意力架构之一。
  - 劣势：工程实现复杂度更高，需要配套的压缩/还原线性层设计。
- **代表落地模型**：Llama 3.1 长上下文版本、DeepSeek V2/V3、Qwen2 超长上下文版本。

> 另外**所有优化方案，仅改变KV的头数/维度，不改变最终输出形状**。MQA/GQA/MLA的最终输出形状仍为`[B, S, D]`，可无缝替换MHA，无需修改Transformer的其他结构。训练时使用什么注意力架构，推理时就必须用对应的架构，KV Cache的格式完全由注意力架构决定。
> 
> FlashAttention的核心是分块计算解决SRAM装不下的问题，而MQA/GQA/MLA是从架构上减少KV的数据量，两者是互补关系，工业界普遍是「GQA + FlashAttention」组合使用，实现长上下文推理的极致加速。
### 3.2.4 实现思路
> 没引入ROPE，不太完整的最小实现

In [9]:
import sys
from pathlib import Path
# 把项目根目录加入 Python 路径
ROOT = Path("..").absolute()
sys.path.append(str(ROOT))

import torch
import torch.nn as nn
import import_ipynb
from transformer.transformer import TransformerLinear

class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 定义线性层
        self.q_proj = TransformerLinear(d_model, d_model, bias=False)
        self.k_proj = TransformerLinear(d_model, d_model, bias=False)
        self.v_proj = TransformerLinear(d_model, d_model, bias=False)
        self.out_proj = TransformerLinear(d_model, d_model)

    def forward(self, x:torch.Tensor):
        B, S, D = x.shape
        H, Dh = self.num_heads, self.d_k

        # view的作用是把最后一个维度 d_model 切分成 num_heads 个头，每个头的维度是 d_k
        # transpose的作用是把维度调整成 [Batch, num_heads, SeqLen, d_k]，方便后续计算
        # 维度变化是 [B, S, D] -> [B, S, H, Dh] -> [B, H, S, Dh]
        q = self.q_proj(x).view(B, S, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, S, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, H, Dh).transpose(1, 2)

        # tril函数生成一个下三角矩阵，表示因果 mask
        mask = torch.tril(torch.ones(S, S, dtype=torch.bool, device=x.device))

        output, attn = scaled_dot_product_attention(q, k, v, mask=mask)
        # 把多头的输出拼接回 [Batch, SeqLen, d_model]
        # contiguous的作用是确保内存连续，view才能正确工作
        # 维度变化是 [B, H, S, Dh] -> [B, S, H, Dh] -> [B, S, D]
        output = output.transpose(1, 2).contiguous().view(B, S, D)
        # 最后通过线性层得到最终输出
        output = self.out_proj(output)
        return output, attn

## 3.3 位置编码
前面我们说的已经很详细了，**纯 attention 本身对顺序并不敏感**。而语言是强顺序结构，所以必须补位置信息。
### 3.3.1 从相加到旋转
早期Transformer（如GPT-2、BERT）的位置编码方案是**绝对位置编码**：预定义一组位置向量，直接加到Token Embedding上输入模型。这种方案虽然能用，但存在三个天然硬伤：
1.  **位置信息是“外挂”的**：位置向量和语义向量简单相加，位置没有真正融入语义交互的核心逻辑。
2.  **相对位置依赖需要模型“硬学”**：模型需要从训练数据中自己摸索“第m个token和第n个token的相对距离是m-n”，不够优雅且数据效率低。
3.  **长上下文外推性差**：训练时只见过S长度的位置，推理时遇到S+1的位置，模型很难自然泛化。

RoPE（Rotary Position Embedding，旋转位置编码）的设计思路则完全不同，它的核心洞察是：
> **不再把位置信息直接加到输入上，而是直接把位置编码作用在注意力的核心——Query和Key向量上，通过“旋转”来编码位置。**

RoFormer论文明确指出：RoPE通过旋转矩阵编码**绝对位置**，同时在自注意力的点积计算中**自然体现相对位置依赖**，兼具序列长度灵活性、相对距离衰减等优良性质。这种设计让位置不再是“附加属性”，而是直接进入“谁该关注谁”的打分过程，是位置编码方案的一次范式升级。

### 3.3.2 